# 08 — Gate Composition as Geometry

Every sequence of single-qubit gates is a **composition of rotations on the Bloch sphere**.

This geometric perspective lets you:
- Simplify gate sequences
- Understand why certain identities hold (e.g. $HXH = Z$)
- Build intuition for circuit depth and error accumulation


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import draw_bloch_sphere, plot_rotation_path
setup_notebook()

import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

def get_statevector(qc):
    sim = AerSimulator(method="statevector")
    qc2 = qc.copy(); qc2.save_statevector()
    return np.array(sim.run(qc2.decompose()).result().get_statevector())

def bloch(sv):
    a, b = sv[0], sv[1]
    return np.array([2*(a.conj()*b).real, 2*(a.conj()*b).imag, abs(a)**2-abs(b)**2])


## Visualising a gate sequence step by step

Let's apply $H$, then $T$, then $H$ to $|0\rangle$ and watch the Bloch vector move.


In [ ]:
gate_sequence = [
    ("H",  lambda qc: qc.h(0)),
    ("T",  lambda qc: qc.t(0)),
    ("H",  lambda qc: qc.h(0)),
]

# Compute intermediate Bloch vectors
bloch_path = []
qc = QuantumCircuit(1)
bloch_path.append(bloch(get_statevector(qc)))   # initial |0>

for name, gate_fn in gate_sequence:
    gate_fn(qc)
    bv = bloch(get_statevector(qc))
    bloch_path.append(bv)
    print(f"After {name}: Bloch = ({bv[0]:+.4f}, {bv[1]:+.4f}, {bv[2]:+.4f})")

bloch_path = np.array(bloch_path)

fig = plt.figure(figsize=(5, 5))
ax = fig.add_subplot(111, projection="3d")
plot_rotation_path(bloch_path, ax=ax, title="HTH applied to |0⟩ step by step")
plt.tight_layout()
plt.show()


## Gate identities as geometric facts

The identity $H X H = Z$ has a clean geometric explanation:
- $X$ is a 180° rotation around the X axis
- $H$ is a 180° rotation around the X+Z diagonal
- Conjugating X by H maps the X axis to the Z axis → XH → Z

Let's verify numerically.


In [ ]:
def circuit_unitary(gates):
    """Compute the 2×2 unitary for a sequence of gates on 1 qubit."""
    from qiskit.quantum_info import Operator
    qc = QuantumCircuit(1)
    for g in gates:
        g(qc)
    return np.array(Operator(qc).data)

X_gate = np.array([[0,1],[1,0]], dtype=complex)
Z_gate = np.array([[1,0],[0,-1]], dtype=complex)

HXH = circuit_unitary([
    lambda qc: qc.h(0),
    lambda qc: qc.x(0),
    lambda qc: qc.h(0),
])
print("HXH =")
print(np.round(HXH, 4))
print()
print("Z =")
print(np.round(Z_gate, 4))
print()
phase_ratio = HXH[0,1] / Z_gate[0,1]  # compare off-diagonal
print(f"Global phase factor: {phase_ratio:.4f}  (|phase| should be 1.0)")


## The Euler angle decomposition

Any single-qubit unitary can be decomposed as $R_z(\alpha) R_y(\beta) R_z(\gamma)$ (ZYZ decomposition).

This means every circuit of arbitrary depth can always be replaced by **at most 3 rotations**.


In [ ]:
# Demonstrate: H T H T H is still a single rotation
def multi_gate_circuit():
    qc = QuantumCircuit(1)
    qc.h(0); qc.t(0); qc.h(0); qc.t(0); qc.h(0)
    return qc

from qiskit.quantum_info import Operator
U = np.array(Operator(multi_gate_circuit()).data)
print("HTHTH unitary:")
print(np.round(U, 4))
print()
print(f"det(U) = {np.linalg.det(U):.6f}  (should be ≈ 1+0i for SU(2))")
print(f"U†U    =")
print(np.round(U.conj().T @ U, 4))


## Summary

- Every gate sequence is a composition of rotations on the Bloch sphere.
- Gate identities like $HXH = Z$ have clean geometric explanations.
- The ZYZ decomposition proves any circuit can be reduced to at most 3 rotations.
- `rqm-core` SU(2) composition handles this algebra directly.

**Next:** [09_ibm_ready_path.ipynb](09_ibm_ready_path.ipynb)
